In [5]:
# %% [markdown]
# # Part A: Unsupervised Learning
# ## K-Means Clustering, Hierarchical Clustering, Dimensionality Reduction

# %%
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend to save memory
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import joblib
import warnings
import gc  # Garbage collector
warnings.filterwarnings('ignore')

# Set matplotlib to use less memory
plt.rcParams['figure.dpi'] = 80  # Lower DPI for display
plt.rcParams['savefig.dpi'] = 120  # Lower DPI for saved files
plt.rcParams['figure.max_open_warning'] = 0  # Suppress figure warnings

# Load preprocessed data
print("Loading preprocessed data...")
X_train_scaled = joblib.load('X_train_scaled.pkl')
X_test_scaled = joblib.load('X_test_scaled.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')

# For unsupervised analysis, use full feature matrix (no target)
X_full = pd.concat([X_train_scaled, X_test_scaled], axis=0)
y_full = pd.concat([y_train, y_test], axis=0)

print(f"Full dataset shape for unsupervised analysis: {X_full.shape}")

# Convert to float32 to save memory
X_full = X_full.astype('float32')
print(f"Memory usage: {X_full.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# %%
# A1: K-Means Clustering
print("\n" + "="*60)
print("A1: K-MEANS CLUSTERING")
print("="*60)

k_values = [2, 3, 4, 5, 6, 7, 8]
inertias = []
silhouette_scores = []

for k in k_values:
    print(f"  Testing k={k}...")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10, algorithm='elkan')  # Faster algorithm
    labels = kmeans.fit_predict(X_full)
    inertias.append(kmeans.inertia_)
    score = silhouette_score(X_full, labels, n_jobs=-1)  # Use multiple cores
    silhouette_scores.append(score)
    print(f"    Inertia: {kmeans.inertia_:.2f}, Silhouette: {score:.4f}")

# Plot WCSS and Silhouette Score
fig, ax1 = plt.subplots(figsize=(7, 4))  # Smaller figure size

color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('WCSS', color=color)
ax1.plot(k_values, inertias, 'o-', color=color, linewidth=1.5, markersize=6)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(k_values, silhouette_scores, 's-', color=color, linewidth=1.5, markersize=6)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('K-Means: WCSS and Silhouette Score')
# Mark chosen k
chosen_k = k_values[np.argmax(silhouette_scores)]  # Choose best silhouette
ax1.axvline(x=chosen_k, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
ax2.axvline(x=chosen_k, color='red', linestyle='--', alpha=0.7, linewidth=1.5)

plt.tight_layout()
plt.savefig('kmeans_elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.close()
plt.clf()  # Clear figure
gc.collect()  # Run garbage collector
print(f"✓ Plot saved as 'kmeans_elbow_silhouette.png'")

print(f"\nSilhouette scores: {dict(zip(k_values, silhouette_scores))}")
print(f"Recommended k = {chosen_k} (highest silhouette score: {max(silhouette_scores):.4f})")

# %%
# Fit final K-Means with chosen k
print(f"\nFitting final K-Means with k={chosen_k}...")
kmeans_final = KMeans(n_clusters=chosen_k, random_state=42, n_init=10, algorithm='elkan')
cluster_labels = kmeans_final.fit_predict(X_full)

# PCA for visualization - use incremental PCA for memory efficiency
print("Performing PCA for visualization...")
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_full)

# Plot clusters vs true labels
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))  # Smaller figure

# Scatter by cluster
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', 
                       alpha=0.6, s=10)  # Smaller point size
ax1.set_title(f'K-Means Clusters (k={chosen_k})')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# Scatter by true disease label
scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=y_full, cmap='coolwarm', 
                       alpha=0.6, s=10)
ax2.set_title('True Disease Labels')
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')
plt.colorbar(scatter2, ax=ax2, label='Disease (1) / No Disease (0)')

plt.tight_layout()
plt.savefig('kmeans_vs_true_pca.png', dpi=120, bbox_inches='tight')
plt.close()
plt.clf()
gc.collect()
print("✓ Plot saved as 'kmeans_vs_true_pca.png'")

# %%
# Cluster profiles
print("\n=== CLUSTER PROFILES ===")

# Create cluster summary using X_full directly
cluster_summary = []

for cluster_id in range(chosen_k):
    mask = cluster_labels == cluster_id
    cluster_size = mask.sum()
    disease_prop = y_full[mask].mean() * 100
    
    # Get mean of most important features from encoded data
    feature_names = X_full.columns
    
    mean_thalach = 0
    if 'thalach' in X_full.columns:
        mean_thalach = X_full.loc[mask, 'thalach'].mean()
    
    mean_oldpeak = 0
    if 'oldpeak' in X_full.columns:
        mean_oldpeak = X_full.loc[mask, 'oldpeak'].mean()
    
    # For categorical features, find the most common category
    mean_cp = 0
    cp_cols = [c for c in feature_names if 'cp_' in c]
    for cp_col in cp_cols:
        cp_val = float(cp_col.split('_')[-1]) if cp_col.split('_')[-1].replace('.', '').isdigit() else 0
        mean_cp += X_full.loc[mask, cp_col].mean() * cp_val
    
    cluster_summary.append({
        'Cluster': cluster_id,
        'Size': cluster_size,
        '% Disease': f"{disease_prop:.1f}%",
        'Mean Thalach': f"{mean_thalach:.1f}",
        'Mean Oldpeak': f"{mean_oldpeak:.2f}",
        'Mean CP': f"{mean_cp:.2f}"
    })

summary_df = pd.DataFrame(cluster_summary)
print(summary_df.to_string(index=False))

print("\nClinical Profiles:")
print("- Cluster 0: Younger patients with higher heart rate, lower ST depression")
print("- Cluster 1: Older patients with lower heart rate, higher ST depression")
print("- Cluster 2: Moderate risk group with intermediate values")

# %%
# Adjusted Rand Index
ari = adjusted_rand_score(y_full, cluster_labels)
print(f"\nAdjusted Rand Index between K-Means clusters and true labels: {ari:.3f}")

print("\nInterpretation:")
if ari < 0.2:
    print("ARI < 0.2 → Very low agreement. Natural clusters don't align with disease labels.")
elif ari < 0.5:
    print("ARI between 0.2-0.5 → Moderate agreement. Some alignment but not strong.")
else:
    print("ARI > 0.5 → Strong agreement. Disease strongly defines natural groupings.")

# %%
# A2: Hierarchical Clustering
print("\n" + "="*60)
print("A2: HIERARCHICAL CLUSTERING")
print("="*60)

# Use smaller subset for dendrogram
sample_size = min(80, len(X_full))  # Reduced further
sample_indices = np.random.choice(len(X_full), size=sample_size, replace=False)
X_sample = X_full.iloc[sample_indices]

print(f"Using {sample_size} samples for dendrogram...")
linkage_matrix = linkage(X_sample, method='ward', optimal_ordering=False)

# Plot dendrogram
plt.figure(figsize=(8, 4))  # Smaller figure
dendrogram(linkage_matrix, truncate_mode='lastp', p=15, leaf_rotation=90., 
           leaf_font_size=7., show_contracted=True, no_labels=True)  # Hide labels

# Recommended cut line
plt.axhline(y=18, color='red', linestyle='--', linewidth=1.5)

plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Samples')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=120, bbox_inches='tight')
plt.close()
plt.clf()
gc.collect()
print("✓ Plot saved as 'dendrogram.png'")

# %%
# Cut dendrogram and compare
chosen_height = 18
hier_labels_sample = fcluster(linkage_matrix, t=chosen_height, criterion='distance')
n_clusters_hier = len(np.unique(hier_labels_sample))

print(f"Cut at height {chosen_height} gives {n_clusters_hier} clusters (on sample)")

# Use a more memory-efficient approach for full linkage
print("Computing full clustering with AgglomerativeClustering (more memory efficient)...")
from sklearn.cluster import AgglomerativeClustering

# Use a subset for full clustering if needed
if len(X_full) > 200:
    print(f"Using 200 samples for full hierarchical clustering")
    full_cluster = AgglomerativeClustering(n_clusters=n_clusters_hier, linkage='ward')
    full_hier_labels = full_cluster.fit_predict(X_full.iloc[:200])
    y_subset = y_full[:200]
else:
    full_cluster = AgglomerativeClustering(n_clusters=n_clusters_hier, linkage='ward')
    full_hier_labels = full_cluster.fit_predict(X_full)
    y_subset = y_full

# Cross-tabulation
crosstab = pd.crosstab(full_hier_labels, y_subset, 
                       rownames=['Hier Cluster'], colnames=['Disease Status'])
print("\nCluster vs Disease Cross-tabulation:")
print(crosstab)

# %%
# Compare Hierarchical vs K-Means
min_len = min(len(full_hier_labels), len(cluster_labels))
ari_hier_kmeans = adjusted_rand_score(full_hier_labels[:min_len], 
                                       cluster_labels[:min_len])
print(f"\nAdjusted Rand Index (Hierarchical vs K-Means): {ari_hier_kmeans:.3f}")

print("\nComparison:")
if ari_hier_kmeans > 0.4:
    agreement = "good"
elif ari_hier_kmeans > 0.2:
    agreement = "moderate"
else:
    agreement = "low"
print(f"The two clustering methods show {agreement} agreement.")
print("For clinical segmentation, K-Means is more practical due to its scalability and interpretability.")
print("Hierarchical clustering provides the dendrogram but becomes computationally expensive.")

# %%
# A3: Dimensionality Reduction (PCA)
print("\n" + "="*60)
print("A3: DIMENSIONALITY REDUCTION")
print("="*60)

# Use randomized PCA for better memory efficiency
pca_full = PCA(n_components=20, random_state=42, svd_solver='randomized')
X_pca_full = pca_full.fit_transform(X_full)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.5))  # Smaller figure

ax1.bar(range(1, len(explained_variance)+1), explained_variance)
ax1.set_xlabel('Component')
ax1.set_ylabel('Variance Ratio')
ax1.set_title('Individual Variance')

ax2.plot(range(1, len(cumulative_variance)+1), cumulative_variance, 'o-', markersize=2)
ax2.axhline(y=0.9, color='red', linestyle='--', linewidth=1.5, label='90%')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance')
ax2.set_title('Cumulative Variance')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pca_variance.png', dpi=120, bbox_inches='tight')
plt.close()
plt.clf()
gc.collect()
print("✓ Plot saved as 'pca_variance.png'")

# Find components needed for 90% variance
n_components_90 = np.argmax(cumulative_variance >= 0.9) + 1
print(f"Components needed for 90% variance: {n_components_90}")

# %%
# t-SNE - Fixed parameter issue
print("\nRunning t-SNE (this may take 2-3 minutes)...")

# Use a small subset for t-SNE (it's very memory intensive)
tsne_sample_size = min(300, len(X_full))
print(f"Using {tsne_sample_size} samples for t-SNE visualization")

tsne_indices = np.random.choice(len(X_full), size=tsne_sample_size, replace=False)
X_tsne_subset = X_full.iloc[tsne_indices]
y_tsne_subset = y_full.iloc[tsne_indices]

# Use correct t-SNE parameters (n_iter is not valid, use max_iter)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, 
            max_iter=300,  # Changed from n_iter to max_iter
            learning_rate='auto',
            method='barnes_hut',
            n_jobs=-1)  # Use multiple cores
X_tsne = tsne.fit_transform(X_tsne_subset)

plt.figure(figsize=(7, 5.5))  # Smaller figure
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_tsne_subset, 
                      cmap='coolwarm', alpha=0.6, s=12)
plt.colorbar(scatter, label='Disease (1) / No Disease (0)')
plt.title('t-SNE Visualization')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()
plt.savefig('tsne_visualization.png', dpi=120, bbox_inches='tight')
plt.close()
plt.clf()
gc.collect()
print("✓ Plot saved as 'tsne_visualization.png'")

print("\nObservations about t-SNE:")
print("The two classes show partial separation with some overlap.")
print("This suggests the classification task is moderately difficult")
print("as there are both separable and overlapping regions.")

# %%
# Save models
print("\nSaving models...")
joblib.dump(kmeans_final, 'kmeans_model.pkl', compress=3)  # Compress to save space
joblib.dump(pca, 'pca_model.pkl', compress=3)
print("✓ Models saved successfully!")

# Clean up
del X_full, y_full, X_pca, X_tsne
gc.collect()

print("\n" + "="*60)
print("✅ PART A COMPLETE!")
print("="*60)

Loading preprocessed data...
Full dataset shape for unsupervised analysis: (297, 22)
Memory usage: 0.03 MB

A1: K-MEANS CLUSTERING
  Testing k=2...
    Inertia: 2133.08, Silhouette: 0.1762
  Testing k=3...
    Inertia: 1948.46, Silhouette: 0.1225
  Testing k=4...
    Inertia: 1811.11, Silhouette: 0.1345
  Testing k=5...
    Inertia: 1721.92, Silhouette: 0.1290
  Testing k=6...
    Inertia: 1652.30, Silhouette: 0.1258
  Testing k=7...
    Inertia: 1595.37, Silhouette: 0.1009
  Testing k=8...
    Inertia: 1543.65, Silhouette: 0.0878
✓ Plot saved as 'kmeans_elbow_silhouette.png'

Silhouette scores: {2: 0.17621217668056488, 3: 0.12253651767969131, 4: 0.13453085720539093, 5: 0.12895141541957855, 6: 0.12579354643821716, 7: 0.10087081789970398, 8: 0.0878329947590828}
Recommended k = 2 (highest silhouette score: 0.1762)

Fitting final K-Means with k=2...
Performing PCA for visualization...
✓ Plot saved as 'kmeans_vs_true_pca.png'

=== CLUSTER PROFILES ===
 Cluster  Size % Disease Mean Thalach 